# VisageAI - Training on Google Colab
Notebook này giúp bạn dễ dàng thực hiện quá trình training mô hình AI Face Rating trên Google Colab với GPU.

**Lưu ý quan trọng trước khi bắt đầu:**
1. Vào **Runtime > Change runtime type** và chọn **T4 GPU** (hoặc GPU khác nếu có) để tăng tốc độ chạy DINOv2.
2. Đảm bảo bạn đã upload file `SCUT-FBP5500_v2.zip` lên Google Drive cá nhân của mình, chính xác tại đường dẫn: `MyDrive/Datasets/SCUT-FBP5500_v2.zip`

## 1. Mount Google Drive
Kết nối với Google Drive để có thể lấy dataset và lưu lại weights sau khi train.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone Repository và Cài đặt thư viện
Tải code từ GitHub và cài đặt các thư viện cần thiết.

In [ ]:
!git clone https://github.com/ChienPM-27/VisageAI.git
%cd VisageAI
!pip install -r requirements.txt -q

# Kiểm tra GPU
!nvidia-smi

## 3. Giải nén Dataset từ Google Drive
Giải nén file zip từ `MyDrive/Datasets/SCUT-FBP5500_v2.zip` vào thư mục `data/` của project.

In [ ]:
!mkdir -p data
!unzip -q /content/drive/MyDrive/Datasets/SCUT-FBP5500_v2.zip -d data/

# Liệt kê để kiểm tra xem cấu trúc thư mục đã đúng chưa
!ls data/SCUT-FBP5500_v2/

## 4. Trích xuất đặc trưng (Feature Extraction)
Chạy file `extract_features.py` để trích xuất Landmarks từ MediaPipe và Embedding từ DINOv2.
Kết quả sẽ được lưu vào `data/dataset.npz`.

In [ ]:
!python src/scripts/extract_features.py \
    --img_dir data/SCUT-FBP5500_v2/Images \
    --labels data/SCUT-FBP5500_v2/train_test_files/All_Labels.txt \
    --output data/dataset.npz \
    --pool cls

## 5. Train MLP Regressor
Huấn luyện mạng Neural Network để dự đoán điểm thẩm mỹ dựa trên các feature vừa trích xuất.

In [ ]:
!python src/training/train_mlp.py \
    --dataset data/dataset.npz \
    --epochs 150 \
    --batch_size 64 \
    --alpha 0.5 \
    --beta 0.2 \
    --out_dir weights

## 6. Train Anchors (K-Means Medoids)
Trích xuất các khuôn mặt đại diện cho từng tier thẩm mỹ từ tập training.

In [ ]:
!python src/training/train_anchors.py \
    --dataset data/dataset.npz \
    --out_dir weights \
    --n_anchors 5

## 7. Sao lưu Weights về Google Drive
Copy thư mục `weights/` (bao gồm model, scaler, anchors, JSON meta) trở về Google Drive để sử dụng ở môi trường local hoặc server của bạn.

In [ ]:
!mkdir -p /content/drive/MyDrive/VisageAI_Weights
!cp -r weights/* /content/drive/MyDrive/VisageAI_Weights/
print("\nĐã lưu weights thành công vào thư mục: MyDrive/VisageAI_Weights/")
print("Bạn có thể tải thư mục này về và đặt vào folder 'weights/' trong source code local.")